# VectorStoreIndex — Your First LlamaIndex RAG Pipeline (Step by Step)

Episode 1 built a working RAG pipeline in under 10 lines, but glossed over what each step actually does. This episode slows down and walks through documents → nodes → index → query engine explicitly, using the same `VectorStoreIndex` from before.


First, some setup: configure logging so the output stays readable, load API keys from `.env`, and tell LlamaIndex which LLM and embedding model to use for everything below.


In [11]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# LlamaIndex and its HTTP client (httpx) log a lot of INFO-level noise by default —
# bump both up to WARNING so only real problems show up in the output below.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Reads the .env file and copies its keys (e.g. OPENAI_API_KEY) into os.environ,
# which is how the OpenAI clients below pick up your API key.
load_dotenv()

# Settings is a global config object — every index and query engine we build in
# this notebook will use these two models unless we override them explicitly.
Settings.llm = OpenAI(model="gpt-4.1-nano")  # the LLM that reads context and writes answers
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")  # turns text into vectors

**Step 1 — Load documents.** `SimpleDirectoryReader` reads every file in `data/sample_docs` (our five anime overview `.txt` files) and wraps each one in a `Document` object, LlamaIndex's basic container for raw text plus metadata like the filename.


In [12]:
from llama_index.core import SimpleDirectoryReader

# Reads every file in the folder and wraps each one in a Document — LlamaIndex's
# container for a chunk of raw text plus metadata (filename, etc).
documents = SimpleDirectoryReader("data/sample_docs").load_data()
print(f"Loaded {len(documents)} documents")

# Peek at the first 200 characters of the first document to confirm the text
# actually made it in correctly.
print(f"\nFirst document snippet:\n{documents[0].text[:200]}")

Loaded 5 documents

First document snippet:
Death Note — Series Overview

Overview
Death Note is a Japanese manga series written by Tsugumi Ohba and illustrated by Takeshi Obata, serialized in Weekly Shonen Jump from 2003 to 2006. Unlike most S


**Step 2 — Build the index.** `VectorStoreIndex.from_documents()` does three things in one call: splits each `Document` into smaller `Node` chunks, calls the embedding model on every chunk to turn it into a vector, and stores those vectors in memory so they can be searched by meaning later.


In [13]:
from llama_index.core import VectorStoreIndex

# from_documents() chunks each Document into Nodes, embeds every Node with
# Settings.embed_model, and stores the resulting vectors — all in one call.
index = VectorStoreIndex.from_documents(documents)
print(f"Index built from {len(index.docstore.docs)} nodes (chunks)")

Index built from 8 nodes (chunks)


**Step 3 — Query it.** `index.as_query_engine()` wraps the index with a retriever (finds the most relevant nodes for a question) and a response synthesizer (asks the LLM to answer using those nodes) so the whole retrieve-then-answer flow happens in a single `.query()` call.


In [14]:
# as_query_engine() bundles a retriever and a response synthesizer together so
# retrieval + LLM synthesis happen behind a single .query() call.
query_engine = index.as_query_engine()

questions = [
    "What is Naruto's signature technique?",
    "How many Dragon Balls are needed to summon Shenron?",
    "What happens if no cause of death is specified in the Death Note?",
]

responses = []  # keep every response object so we can inspect its sources afterward
for question in questions:
    response = query_engine.query(question)  # retrieves relevant nodes, then asks the LLM to answer
    responses.append(response)
    print(f"Q: {question}\nA: {response}\n")

Q: What is Naruto's signature technique?
A: Naruto's signature technique is the Rasengan, a swirling ball of concentrated chakra.

Q: How many Dragon Balls are needed to summon Shenron?
A: Seven Dragon Balls are needed to summon Shenron.

Q: What happens if no cause of death is specified in the Death Note?
A: If no cause of death is specified in the Death Note, the person dies of a heart attack exactly 40 seconds after their name is written.



**Step 4 — Prove the answer is grounded.** Every `Response` object keeps track of the exact nodes the LLM was given as context. Printing `source_nodes` for the last answer shows _why_ the model said what it said — this is what separates RAG from an LLM just making things up.


In [15]:
# Every response carries the exact nodes the LLM used as context — this is how
# you verify an answer is grounded in your documents rather than made up.
print("Source nodes used for the last answer:\n")
for source_node in responses[-1].source_nodes:
    print("---")
    print(f"Score: {source_node.score:.3f}")  # similarity score between the question and this node
    print(f"Text: {source_node.node.get_content()[:300]}\n")  # the actual chunk text the LLM read

Source nodes used for the last answer:

---
Score: 0.597
Text: Death Note — Series Overview

Overview
Death Note is a Japanese manga series written by Tsugumi Ohba and illustrated by Takeshi Obata, serialized in Weekly Shonen Jump from 2003 to 2006. Unlike most Shonen Jump series, it's a psychological thriller rather than an action-adventure story, centered on 

---
Score: 0.269
Text: Demon Slayer — Series Overview

Overview
Demon Slayer (Kimetsu no Yaiba) is a Japanese manga series written and illustrated by Koyoharu Gotouge, serialized in Weekly Shonen Jump from 2016 to 2020. Set in Taisho-era Japan, it follows Tanjiro Kamado, a kind-hearted boy who becomes a demon slayer after



### Summary

- **Documents** are your raw source files; **Nodes** are the chunks LlamaIndex splits them into.
- The **Index** stores embedded nodes so they can be searched by meaning, not just keywords.
- The **Query Engine** ties retrieval and LLM synthesis together into a single `.query()` call — and its `source_nodes` prove every answer is grounded in your actual documents.
